In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('/workspaces/practice_models/files/football/results.csv')
print(df.head())
print(df.columns.tolist())

         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1  1873-03-08   England  Scotland           4           2   Friendly   London   
2  1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3  1875-03-06   England  Scotland           2           2   Friendly   London   
4  1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


In [3]:
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)
print(df['home_win'])

0        0
1        1
2        1
3        0
4        1
        ..
49066    0
49067    0
49068    0
49069    0
49070    0
Name: home_win, Length: 49071, dtype: int64


In [4]:
df['away_win'] = (df['away_score'] > df['home_score']).astype(int)
print(df['away_win'])

0        0
1        0
2        0
3        0
4        0
        ..
49066    0
49067    1
49068    1
49069    1
49070    0
Name: away_win, Length: 49071, dtype: int64


In [5]:
df['draw'] = (df['home_score'] == df['away_score']).astype(int)
print(df['draw'])

0        1
1        0
2        0
3        1
4        0
        ..
49066    1
49067    0
49068    0
49069    0
49070    1
Name: draw, Length: 49071, dtype: int64


In [6]:
df['goal_difference'] = df['home_score'] - df['away_score']
print(df['goal_difference'])

0        0
1        2
2        1
3        0
4        3
        ..
49066    0
49067   -1
49068   -1
49069   -1
49070    0
Name: goal_difference, Length: 49071, dtype: int64


In [7]:
home_counts = df.groupby('home_team').size().reset_index(name='home_games_played')
df = df.merge(home_counts, on='home_team', how='left')

In [8]:
away_counts = df.groupby('away_team').size().reset_index(name='away_games_played')
df = df.merge(away_counts, on='away_team', how='left')

In [9]:
home_avg = df.groupby('home_team')['home_score'].mean().reset_index()
home_avg.columns = ['home_team', 'home_avg_goals']
df = df.merge(home_avg, on='home_team', how='left')
print(df[['home_team', 'home_win', 'home_avg_goals']].head(10))

  home_team  home_win  home_avg_goals
0  Scotland         0        1.898305
1   England         1        2.287546
2  Scotland         1        1.898305
3   England         0        2.287546
4  Scotland         1        1.898305
5  Scotland         1        1.898305
6   England         0        2.287546
7     Wales         0        1.464986
8  Scotland         1        1.898305
9  Scotland         1        1.898305


In [10]:
away_avg = df.groupby('away_team')['away_score'].mean().reset_index()
away_avg.columns = ['away_team', 'away_avg_goals']
df = df.merge(away_avg, on='away_team', how='left')
print(df[['away_team', 'away_win', 'away_avg_goals']].head(10))

  away_team  away_win  away_avg_goals
0   England         0        2.087037
1  Scotland         0        1.525346
2   England         0        2.087037
3  Scotland         0        1.525346
4   England         0        2.087037
5     Wales         0        1.049315
6  Scotland         1        1.525346
7  Scotland         1        1.525346
8   England         0        2.087037
9     Wales         0        1.049315


In [11]:
tournaments_type = df['tournament'].unique()
print(tournaments_type)
print(f'Number of unique tournaments: {len(tournaments_type)}')


<StringArray>
[                        'Friendly',        'British Home Championship',
             'Évence Coppée Trophy',                     'Muratti Vase',
                      'Copa Lipton',                      'Copa Newton',
      'Copa Premio Honor Argentino',                    'Olympic Games',
       'Copa Premio Honor Uruguayo',   'Far Eastern Championship Games',
 ...
               'Tri-Nations Series', 'ASEAN Championship qualification',
               'ASEAN Championship',  'EAFF Championship qualification',
                    'Mapinduzi Cup',                  'Canadian Shield',
          'Outrigger Challenge Cup',            'South Asian Super Cup',
                  'CONCACAF Series',         'Al Ain International Cup']
Length: 191, dtype: str
Number of unique tournaments: 191


In [12]:
neutral_check = df['neutral'].unique()
print(neutral_check)
print(f'Number of neutrality: {len(neutral_check)}')


[False  True]
Number of neutrality: 2


In [13]:
features = ['home_avg_goals', 'home_games_played', 'away_avg_goals', 'away_games_played', 'draw', 'neutral']
X = df[features].dropna()
Y = df.loc[X.index, 'home_win']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=60)

model = LogisticRegression()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(Y_test, Y_pred):.2%}')

Accuracy: 78.49%


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
from sklearn.ensemble import RandomForestClassifier


features = ['home_avg_goals', 'home_games_played', 'away_avg_goals', 'away_games_played', 'draw', 'neutral']
X = df[features].dropna()
Y = df.loc[X.index, 'home_win']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=60)

rf_model = RandomForestClassifier()
rf_model.fit(X_train, Y_train)

Y_pred = rf_model.predict(X_test)
print(f'Accuracy: {accuracy_score(Y_test, Y_pred):.2%}')

Accuracy: 78.45%


In [16]:
for feature, coef in zip(features, model.coef_[0]):
  print(f'{feature}: {coef:.3f}')

home_avg_goals: 1.478
home_games_played: 0.002
away_avg_goals: -1.927
away_games_played: -0.003
draw: -7.562
neutral: -0.532


In [17]:
england_home_avg_goals = df[df['home_team'] == 'England']['home_avg_goals'].iloc[0]
england_home_games_played = df[df['home_team'] == 'England']['home_games_played'].iloc[0]

brazil_away_avg_goals = df[df['away_team'] == 'Brazil']['away_avg_goals'].iloc[0]
brazil_away_games_played = df[df['away_team'] == 'Brazil']['away_games_played'].iloc[0]

prediction_features = {
    'home_avg_goals': [england_home_avg_goals],
    'home_games_played': [england_home_games_played],
    'away_avg_goals': [brazil_away_avg_goals],
    'away_games_played': [brazil_away_games_played],
    'draw': [0],
    'neutral': [0]
}

prediction_df = pd.DataFrame(prediction_features)
predicted_outcome = model.predict(prediction_df)

if predicted_outcome[0] == 1:
    print("Prediction: England (Home Team) is predicted to win.")
else:
    print("Prediction: England (Home Team) is NOT predicted to win (could be a draw or Brazil win).")

Prediction: England (Home Team) is predicted to win.
